# SMART 2 - Data Cleaning

**Goal:** Clean the raw dataset and save a processed version ready for model training.

**Steps:**
1. Load raw data
2. Drop high-null columns (>40% missing)
3. Fill remaining nulls (median for numeric, mode for categorical)
4. Remove outliers using IQR on SalePrice
5. One-hot encode categorical variables (using sklearn for API compatibility)
6. Save cleaned data to `data/processed/`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_processing import (
    load_data, handle_nulls, encode_categoricals,
    remove_outliers
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Raw Data

In [ ]:
df = load_data('../data/raw/train.csv')
print(f'Raw data shape: {df.shape}')

In [ ]:
df.head()

## 2. Drop Id Column

In [ ]:
if 'Id' in df.columns:
    df = df.drop(columns=['Id'])
    print('Dropped Id column')
print(f'Shape after dropping Id: {df.shape}')

## 3. Handle Missing Values

- Drop columns with >40% nulls
- Fill numeric nulls with median
- Fill categorical nulls with mode

In [ ]:
null_report = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df)) * 100
})
null_report = null_report[null_report['null_count'] > 0].sort_values('null_pct', ascending=False)

print(f'Columns with nulls before cleaning: {len(null_report)}')
null_report.head(10)

In [ ]:
df = handle_nulls(df, threshold=0.4)
print(f'Shape after handling nulls: {df.shape}')

In [ ]:
print(f'Total nulls remaining: {df.isnull().sum().sum()}')

## 4. Remove Outliers (SalePrice)

Using IQR method to remove extreme values that could skew the model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot(df['SalePrice'])
axes[0].set_title('SalePrice - Before Outlier Removal')
axes[0].set_ylabel('Price ($)')

Q1 = df['SalePrice'].quantile(0.25)
Q3 = df['SalePrice'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['SalePrice'] < lower) | (df['SalePrice'] > upper)]
print(f'Outliers detected: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)')

df = remove_outliers(df, column='SalePrice', iqr_multiplier=1.5)

axes[1].boxplot(df['SalePrice'])
axes[1].set_title('SalePrice - After Outlier Removal')
axes[1].set_ylabel('Price ($)')

plt.tight_layout()
plt.show()

## 5. Encode Categorical Variables

Using sklearn OneHotEncoder so the same encoder can be reused in the API.

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns to encode: {len(cat_cols)}')
for col in cat_cols:
    print(f'  {col}: {df[col].nunique()} unique values')

In [ ]:
df = encode_categoricals(df)
print(f'Shape after encoding: {df.shape}')

## 6. Final Check

In [ ]:
print(f'Final shape: {df.shape}')
print(f'Nulls remaining: {df.isnull().sum().sum()}')
print(f'Column types: {df.dtypes.value_counts().to_dict()}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['SalePrice'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('SalePrice Distribution (Cleaned)')
axes[0].set_xlabel('Price ($)')

axes[1].hist(np.log1p(df['SalePrice']), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('SalePrice Distribution (Log Transformed)')
axes[1].set_xlabel('Log(Price)')

plt.tight_layout()
plt.show()

## 7. Save Cleaned Data

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/train_clean.csv', index=False)
print(f'Saved cleaned data to data/processed/train_clean.csv')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')